In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import json

PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../output")
CHART_DIR = OUTPUT_DIR / "charts"
EXPORT_DIR = OUTPUT_DIR / "final_exports"

CHART_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SCHOOLS_EXPOSURE_PATH = PROCESSED_DIR / "schools_with_exposure.csv"
DISTRICT_SUMMARY_PATH = PROCESSED_DIR / "district_exposure_summary.csv"
TYPE_SUMMARY_PATH = PROCESSED_DIR / "type_exposure_summary.csv"
LEVEL_SUMMARY_PATH = PROCESSED_DIR / "level_exposure_summary.csv"
TOP_EXPOSED_PATH = PROCESSED_DIR / "top_exposed_schools.csv"
FARTHEST_PATH = PROCESSED_DIR / "farthest_schools.csv"
STATION_COVERAGE_PATH = PROCESSED_DIR / "station_coverage_summary.csv"
CONFIDENCE_PATH = PROCESSED_DIR / "exposure_confidence_summary.csv"

##### Load Tables

In [2]:
schools = pd.read_csv(SCHOOLS_EXPOSURE_PATH)
district_summary = pd.read_csv(DISTRICT_SUMMARY_PATH)
type_summary = pd.read_csv(TYPE_SUMMARY_PATH)
level_summary = pd.read_csv(LEVEL_SUMMARY_PATH)
top_exposed = pd.read_csv(TOP_EXPOSED_PATH)
farthest = pd.read_csv(FARTHEST_PATH)
station_coverage = pd.read_csv(STATION_COVERAGE_PATH)
confidence_summary = pd.read_csv(CONFIDENCE_PATH)

print("schools:", schools.shape)
print("district_summary:", district_summary.shape)
print("type_summary:", type_summary.shape)
print("level_summary:", level_summary.shape)
print("top_exposed:", top_exposed.shape)
print("farthest:", farthest.shape)
print("station_coverage:", station_coverage.shape)
print("confidence_summary:", confidence_summary.shape)

schools: (2423, 61)
district_summary: (16, 9)
type_summary: (2, 9)
level_summary: (16, 7)
top_exposed: (2423, 11)
farthest: (2423, 9)
station_coverage: (44, 7)
confidence_summary: (3, 5)


##### Build KPI Table
This creates the small numbers that usually appear at the top of a dashboard.

In [3]:
kpi = pd.DataFrame([
    {"metric": "total_schools", "value": len(schools)},
    {"metric": "district_count", "value": schools["District"].nunique()},
    {"metric": "station_count_used", "value": schools["nearest_station_id"].nunique()},
    {"metric": "avg_pm25_all", "value": round(schools["PM2.5 (µg/m³)_mean"].mean(), 2)},
    {"metric": "median_pm25_all", "value": round(schools["PM2.5 (µg/m³)_mean"].median(), 2)},
    {"metric": "avg_distance_km", "value": round(schools["distance_km"].mean(), 2)},
    {"metric": "max_distance_km", "value": round(schools["distance_km"].max(), 2)},
    {"metric": "high_confidence_schools", "value": int((schools["exposure_confidence"] == "High").sum())},
    {"metric": "medium_confidence_schools", "value": int((schools["exposure_confidence"] == "Medium").sum())},
    {"metric": "low_confidence_schools", "value": int((schools["exposure_confidence"] == "Low").sum())},
])

kpi

,metric,value
0,total_schools,2423.00
1,district_count,16.00
2,station_count_used,44.00
3,avg_pm25_all,98.04
4,median_pm25_all,97.67
5,avg_distance_km,2.66
6,max_distance_km,15.20
7,high_confidence_schools,2260.00
8,medium_confidence_schools,154.00
9,low_confidence_schools,9.00


##### Build Dashboard-Ready Tables
These are trimmed, smaller tables meant for charts, ranking cards, and dashboard widgets.

In [4]:
chart_districts = district_summary.sort_values("avg_pm25", ascending=False).head(10).copy()
dashboard_type = type_summary.copy()
chart_top_exposed = top_exposed.head(20).copy()
chart_farthest = farthest.head(20).copy()
chart_station_coverage = station_coverage.sort_values("school_count", ascending=False).head(15).copy()
chart_conf = confidence_summary.copy()

print(chart_districts.shape)
print(dashboard_type.shape)
print(chart_top_exposed.shape)
print(chart_farthest.shape)
print(chart_station_coverage.shape)
print(chart_conf.shape)

(10, 9)
(2, 9)
(20, 11)
(20, 9)
(15, 7)
(3, 5)


#####  Export Dashboard CSVs

In [5]:
kpi.to_csv(EXPORT_DIR / "dashboard_kpi.csv", index=False)
chart_districts.to_csv(EXPORT_DIR / "dashboard_district_top10.csv", index=False)
dashboard_type.to_csv(EXPORT_DIR / "dashboard_type_summary.csv", index=False)
level_summary.to_csv(EXPORT_DIR / "dashboard_level_summary.csv", index=False)
chart_top_exposed.to_csv(EXPORT_DIR / "dashboard_top_exposed_top20.csv", index=False)
chart_farthest.to_csv(EXPORT_DIR / "dashboard_farthest_top20.csv", index=False)
chart_station_coverage.to_csv(EXPORT_DIR / "dashboard_station_coverage_top15.csv", index=False)
chart_conf.to_csv(EXPORT_DIR / "dashboard_confidence_summary.csv", index=False)

print("Dashboard CSVs exported")

Dashboard CSVs exported


In [6]:
def save_meta(path, caption, description):
    meta_path = str(path) + ".meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(
            {"caption": caption, "description": description},
            f,
            ensure_ascii=False,
            indent=2
        )

##### Chart 1: Top 10 Districts by PM2.5

In [7]:
fig = px.bar(
    chart_districts,
    x="District",
    y="avg_pm25",
    title="District PM2.5 exposure (top 10)"
)

fig.update_layout(
    title={
        "text": "District PM2.5 exposure (top 10)<br><span style='font-size: 18px; font-weight: normal;'>Average school exposure by district | Higher means worse exposure</span>"
    }
    
)
fig.update_xaxes(title_text="District")
fig.update_yaxes(title_text="PM2.5 mean")
fig.update_traces(cliponaxis=False)

fig.write_image(CHART_DIR / "district_pm25_top10.png")
save_meta(
    CHART_DIR / "district_pm25_top10.png",
    "Top 10 districts by average PM2.5 exposure",
    "Bar chart of the ten districts with the highest average school PM2.5 exposure."
)

##### Chart 2: PM2.5 by School Type

In [8]:
fig = px.bar(
    dashboard_type,
    x="Type",
    y="avg_pm25",
    title="School type PM2.5 exposure"
)

fig.update_layout(
    title={
        "text": "School type PM2.5 exposure<br><span style='font-size: 18px; font-weight: normal;'>Comparison of average PM2.5 exposure across school types</span>"
    }
)
fig.update_xaxes(title_text="School type")
fig.update_yaxes(title_text="PM2.5 mean")
fig.update_traces(cliponaxis=False)

fig.write_image(CHART_DIR / "school_type_pm25.png")
save_meta(
    CHART_DIR / "school_type_pm25.png",
    "Average PM2.5 exposure by school type",
    "Bar chart comparing average PM2.5 exposure for Govt and Unaided schools."
)

##### Chart 3: Exposure Confidence Distribution

In [9]:
fig = px.pie(
    chart_conf,
    names="exposure_confidence",
    values="school_count",
    title="Exposure confidence distribution"
)

fig.update_layout(
    title={
        "text": "Exposure confidence distribution<br><span style='font-size: 18px; font-weight: normal;'>School counts by nearest-station confidence level</span>"
    },
    legend=dict(x=1.02, y=1)
)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.update_layout(uniformtext_minsize=14, uniformtext_mode="hide")

fig.write_image(CHART_DIR / "confidence_distribution.png")
save_meta(
    CHART_DIR / "confidence_distribution.png",
    "School counts by exposure confidence",
    "Pie chart showing High, Medium, and Low confidence groups based on school-to-station distance."
)

##### Chart 4: Top 10 Farthest Schools

In [10]:
plot_farthest = chart_farthest.head(10).copy()
plot_farthest["label"] = plot_farthest["School Name"].str.slice(0, 22)

fig = px.bar(
    plot_farthest,
    x="distance_km",
    y="label",
    orientation="h",
    title="Farthest schools from stations"
)

fig.update_layout(
    title={
        "text": "Farthest schools from stations<br><span style='font-size: 18px; font-weight: normal;'>Top 10 schools with the weakest monitoring proximity</span>"
    }
)
fig.update_xaxes(title_text="Distance km")
fig.update_yaxes(title_text="School")
fig.update_traces(cliponaxis=False)

fig.write_image(CHART_DIR / "farthest_schools_top10.png")
save_meta(
    CHART_DIR / "farthest_schools_top10.png",
    "Top 10 farthest schools from monitoring stations",
    "Horizontal bar chart showing the schools with the greatest distance to their nearest station."
)

##### Chart 5: Station Coverage Load

In [11]:
plot_cov = chart_station_coverage.copy()

fig = px.bar(
    plot_cov,
    x="nearest_station_name",
    y="school_count",
    title="Station coverage load"
)

fig.update_layout(
    title={
        "text": "Station coverage load<br><span style='font-size: 18px; font-weight: normal;'>How many schools are assigned to each major station</span>"
    }
)
fig.update_xaxes(title_text="Station")
fig.update_yaxes(title_text="School count")
fig.update_traces(cliponaxis=False)

fig.write_image(CHART_DIR / "station_coverage_load.png")
save_meta(
    CHART_DIR / "station_coverage_load.png",
    "Top stations by assigned school count",
    "Bar chart showing which pollution stations are assigned the largest number of schools."
)

##### Export Manifest
This is a simple inventory file so you always know what each exported file is for.

In [12]:
export_manifest = pd.DataFrame([
    {"folder": "data/processed", "file": "schools_with_exposure.csv", "purpose": "Master school-level exposure table"},
    {"folder": "data/processed", "file": "district_exposure_summary.csv", "purpose": "District-level exposure summary"},
    {"folder": "data/processed", "file": "type_exposure_summary.csv", "purpose": "School-type exposure summary"},
    {"folder": "data/processed", "file": "level_exposure_summary.csv", "purpose": "School-level category summary"},
    {"folder": "data/processed", "file": "top_exposed_schools.csv", "purpose": "Ranked school exposure list"},
    {"folder": "data/processed", "file": "farthest_schools.csv", "purpose": "Monitoring gap school list"},
    {"folder": "data/processed", "file": "station_coverage_summary.csv", "purpose": "Station assignment load table"},
    {"folder": "data/processed", "file": "exposure_confidence_summary.csv", "purpose": "Confidence bucket summary"},
    {"folder": "output/final_exports", "file": "dashboard_kpi.csv", "purpose": "Dashboard KPI cards source"},
    {"folder": "output/final_exports", "file": "dashboard_district_top10.csv", "purpose": "Top districts chart/table source"},
    {"folder": "output/final_exports", "file": "dashboard_type_summary.csv", "purpose": "School type chart/table source"},
    {"folder": "output/final_exports", "file": "dashboard_level_summary.csv", "purpose": "School level summary for dashboard"},
    {"folder": "output/final_exports", "file": "dashboard_top_exposed_top20.csv", "purpose": "Top exposed schools widget source"},
    {"folder": "output/final_exports", "file": "dashboard_farthest_top20.csv", "purpose": "Monitoring gap widget source"},
    {"folder": "output/final_exports", "file": "dashboard_station_coverage_top15.csv", "purpose": "Station coverage chart/table source"},
    {"folder": "output/final_exports", "file": "dashboard_confidence_summary.csv", "purpose": "Confidence donut/pie source"},
    {"folder": "output/charts", "file": "district_pm25_top10.png", "purpose": "District PM2.5 chart"},
    {"folder": "output/charts", "file": "school_type_pm25.png", "purpose": "School type PM2.5 chart"},
    {"folder": "output/charts", "file": "confidence_distribution.png", "purpose": "Confidence distribution chart"},
    {"folder": "output/charts", "file": "farthest_schools_top10.png", "purpose": "Monitoring gap chart"},
    {"folder": "output/charts", "file": "station_coverage_load.png", "purpose": "Station load chart"},
])

export_manifest.to_csv(EXPORT_DIR / "export_manifest.csv", index=False)
print("Saved export manifest:", EXPORT_DIR / "export_manifest.csv")

Saved export manifest: ../output/final_exports/export_manifest.csv


In [13]:
print("Final export files:")
for p in sorted(EXPORT_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nCharts:")
for p in sorted(CHART_DIR.glob("*.png")):
    print(" -", p.name)

Final export files:
 - dashboard_confidence_summary.csv
 - dashboard_district_top10.csv
 - dashboard_farthest_top20.csv
 - dashboard_kpi.csv
 - dashboard_level_summary.csv
 - dashboard_station_coverage_top15.csv
 - dashboard_top_exposed_top20.csv
 - dashboard_type_summary.csv
 - export_manifest.csv

Charts:
 - confidence_distribution.png
 - district_pm25_top10.png
 - farthest_schools_top10.png
 - school_type_pm25.png
 - station_coverage_load.png
